# Clase 18 presencial: Union-Find y Kruskal

## Pregunta inicial

**¿Cómo conectamos todos los nodos con el menor costo total sin crear ciclos?**

<div style="
    border:1px solid #9bb8d3;
    border-left:6px solid #315f8c;
    background:#eef5fb;
    color:#17212b;
    padding:14px 18px;
    margin:14px 0;
    border-radius:4px;
    line-height:1.55;
">
    <strong style="color:#111111;">
        INSTRUCCIÓN — Cuaderno de trabajo
    </strong>
    <br>
    Predice antes de avanzar, completa las trazas y registra cada respuesta en
    <code style="
        color:#102a43;
        background:#dbeaf5;
        padding:2px 5px;
        border-radius:3px;
        font-weight:600;
    ">notebook.md</code>.
    No respondas dentro de este archivo.
</div>

**Responde esta pregunta en notebook.md.**


## Objetivos de aprendizaje

Al terminar podrás:

- distinguir caminos mínimos y árboles de expansión mínima;
- explicar árbol de expansión, ciclos y la condición V−1;
- mantener componentes con Union-Find;
- implementar `find`, `union`, conectividad, tamaños y contador;
- aplicar compresión de caminos y unión por tamaño;
- ejecutar Kruskal y detectar grafos desconectados;
- validar aristas, pesos negativos, empates y no mutación;
- resolver una versión guiada de CSES Road Reparation;
- comparar cola, deque, heap y Union-Find por su operación dominante;
- diseñar pruebas de invariantes y propiedades del MST.

### Ruta presencial

1. Recuperación y nuevo problema.
2. Expansión, ciclos y componentes.
3. Union-Find y optimizaciones.
4. Kruskal, problemas y pruebas.


## 1. Presentación de la clase

Hasta ahora elegimos el siguiente pendiente para construir rutas desde un origen: BFS usa una cola, 0-1 BFS una deque y Dijkstra un heap. Hoy cambia el producto final. Una red de ciudades ya tiene carreteras posibles con distintos costos y queremos construir un subconjunto que conecte **todas** las ciudades con costo total mínimo.

```text
carreteras baratas → ¿conecta componentes distintas? → aceptar
                    ¿sus extremos ya están conectados? → rechazar ciclo
```

No empezaremos memorizando nombres. Primero descubriremos la consulta repetida; después aparecerá Union-Find y, finalmente, Kruskal.

<div style="border-left:5px solid #315f8c;color:#17212b;background:#eef5fb;padding:12px 16px;margin:14px 0"><strong style="color:#111111;">RUTA — Clase presencial de cuatro horas</strong><br>Predice, ejecuta a mano, compara con el visualizador, implementa y convierte cada invariante en una prueba. Registra respuestas en <code style="
        color:#102a43;
        background:#dbeaf5;
        padding:2px 5px;
        border-radius:3px;
        font-weight:600;
    ">notebook.md</code>..</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué producto final esperamos obtener hoy y en qué se diferencia de un camino desde un origen?

**Responde esta pregunta en notebook.md.**


## 2. Recuperación de estructuras anteriores

| Algoritmo | Pregunta | Estructura auxiliar |
| --- | --- | --- |
| BFS | ¿camino con menor número de aristas? | cola |
| 0-1 BFS | ¿camino mínimo con pesos 0/1? | deque |
| Dijkstra | ¿camino mínimo con pesos no negativos? | heap |
| Kruskal | ¿conectar todo con menor costo total? | Union-Find |

Los tres primeros conservan un origen y producen distancias o predecesores. El cuarto selecciona aristas globalmente: no tiene origen especial ni calcula distancia a cada nodo. La continuidad no está en que los algoritmos resuelvan lo mismo, sino en la pregunta transversal: **¿qué operación se repite y qué estructura la vuelve eficiente?**

<div style="border-left:5px solid #7b4bb7;color:#111111;background:#f4effa;padding:12px 16px;margin:14px 0"><strong style="color:#111111;">IMPORTANTE — Nueva operación dominante</strong><br>Ahora repetiremos: “¿estos dos extremos ya pertenecen a la misma componente?”.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué operación dominante distingue a Kruskal de BFS, 0-1 BFS y Dijkstra?

**Responde esta pregunta en notebook.md.**


## 3. Nueva pregunta: conectar toda la red

Usaremos esta red no dirigida durante toda la clase:

| Carretera | Costo | Carretera | Costo |
| --- | ---: | --- | ---: |
| A–B | 4 | B–D | 3 |
| A–C | 2 | C–D | 1 |
| A–D | 5 | C–E | 6 |
| D–E | 2 | | |

Queremos conectar A, B, C, D y E. No necesitamos todas las carreteras. Una primera intuición es ordenar por costo: C–D, A–C, D–E, B–D, A–B, A–D, C–E. Pero «barata» no basta: una carretera puede ser redundante si sus extremos ya están conectados por las elegidas.

Actividad: marca qué carreteras aceptarías y conserva el costo acumulado. Todavía no uses el nombre del algoritmo.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué no basta aceptar automáticamente todas las carreteras en orden creciente de costo?

**Responde esta pregunta en notebook.md.**


## 4. Dijkstra frente a Kruskal

Considera el triángulo A–B costo 2, A–C costo 2 y B–C costo 1. Un árbol de caminos mínimos desde A puede conservar A–B y A–C: cada destino queda a distancia 2, con costo total de aristas 4. Un árbol de expansión mínima puede usar A–B y B–C, con costo total 3.

Dijkstra optimiza **cada distancia desde A**. Kruskal optimiza **la suma del conjunto que conecta toda la red**. Ejecutar Dijkstra y conservar predecesores responde otro contrato y puede producir una red globalmente más cara.

<div style="border-left:5px solid #b26a00;color:#111111;background:#fff6e6;padding:12px 16px;margin:14px 0"><strong style="color:#111111;">ADVERTENCIA — Dos árboles distintos</strong><br>Un árbol de caminos mínimos desde un origen no es necesariamente un árbol de expansión mínima.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué optimiza cada algoritmo y cuál de ellos necesita un origen?

**Responde esta pregunta en notebook.md.**


## 5. Árboles de expansión

Un **árbol de expansión** es un subgrafo que contiene todos los vértices, usa aristas originales, está conectado y no tiene ciclos. Con V vértices utiliza exactamente V−1 aristas.

La intuición es incremental: comenzamos con V componentes aisladas. Cada arista útil puede reducir el número de componentes como máximo en uno. Se necesitan al menos V−1 uniones efectivas para llegar a una sola componente. Si agregamos otra arista a un grafo ya conectado, aparece un camino previo entre sus extremos y la nueva arista cierra un ciclo.

Para cinco ciudades, la solución final debe contener cuatro carreteras. Esa condición permite detener Kruskal antes de revisar el resto.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué un árbol de expansión conectado con V vértices tiene exactamente V−1 aristas?

**Responde esta pregunta en notebook.md.**


## 6. Por qué evitar ciclos

En un ciclo, cada par de vértices sigue conectado aunque retiremos una de sus aristas. Por tanto, una solución de costo mínimo no necesita conservar el ciclo completo.

```text
A — B
|   |
D — C
```

Antes de aceptar u–v necesitamos responder: «¿ya existe alguna conexión entre u y v dentro del bosque seleccionado?». Podríamos ejecutar BFS o DFS después de cada arista, pero repetir un recorrido del bosque sería costoso. Necesitamos mantener esta respuesta de manera incremental.

<div style="border-left:5px solid #315f8c;color:#111111;background:#eef5fb;padding:12px 16px;margin:14px 0"><strong style="color:#111111;">PUENTE — De ciclos a componentes</strong><br>Si u y v pertenecen a la misma componente, la arista cierra un ciclo; si pertenecen a componentes distintas, las fusiona.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué información mínima necesitamos antes de aceptar una arista u–v?

**Responde esta pregunta en notebook.md.**


## 7. Componentes conexas

Union-Find mantiene una partición: cada elemento pertenece exactamente a un conjunto disjunto.

```text
inicio       {A} {B} {C} {D} {E}
aceptar C–D  {A} {B} {C,D} {E}
aceptar A–C  {A,C,D} {B} {E}
aceptar D–E  {A,C,D,E} {B}
```

El nombre del representante no tiene significado matemático. La componente `{A,C,D}` podría estar representada por A, C o D según el orden y la heurística. Lo importante es que todos produzcan la misma raíz al consultar `find`.

### Comprueba tu comprensión

**Pregunta:** ¿Qué cambia en la partición cuando aceptamos una arista entre componentes distintas?

**Responde esta pregunta en notebook.md.**


## 8. Necesidad de Union-Find

Union-Find —también llamado Disjoint Set Union— ofrece justo las operaciones necesarias:

- `find(x)`: devuelve el representante de x;
- `union(a,b)`: une componentes distintas y devuelve si fue efectiva;
- `conectados(a,b)`: compara representantes;
- `tamano_componente(x)`: consulta el tamaño en la raíz;
- `numero_componentes()`: cuenta las raíces vigentes.

Dentro de Kruskal no necesitamos llamar primero `conectados` y después `union`. El booleano de `union` reúne decisión y actualización: `True` acepta la arista; `False` la rechaza por ciclo.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué conviene que union(a, b) devuelva un booleano?

**Responde esta pregunta en notebook.md.**


## 9. Representación mediante padres

La estructura evaluada usa índices `0,1,…,n−1` y tres atributos:

```python
padre = [0, 1, 2, 3, 4]
tamano = [1, 1, 1, 1, 1]
componentes = 5
```

`padre[i]` señala el siguiente nodo del árbol. Una raíz se apunta a sí misma. Solo `tamano[raiz]` representa el tamaño vigente; valores en nodos que dejaron de ser raíces no deben consultarse. El contador disminuye únicamente en una unión efectiva.

Las ciudades con nombres se traducen mediante una tabla —A→0, B→1, …—, pero Union-Find permanece simple y trabaja con índices.

### Comprueba tu comprensión

**Pregunta:** ¿Qué condición permite reconocer una raíz en el arreglo padre?

**Responde esta pregunta en notebook.md.**


## 10. Operación find

`find(x)` sigue padres hasta encontrar una raíz. En la cadena `4 → 3 → 1 → 0`, `find(4)` devuelve 0.

```text
mientras elemento != padre[elemento]:
    elemento = padre[elemento]
devolver elemento
```

Antes de recorrer se valida el índice. Python acepta índices negativos en listas, pero el contrato de Union-Find no: `find(-1)` debe lanzar `IndexError`, no consultar el último elemento accidentalmente. Tipos incorrectos —incluido `bool`— producen `TypeError`.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué debemos validar explícitamente los índices negativos?

**Responde esta pregunta en notebook.md.**


## 11. Operación union

`union(a,b)` primero encuentra ambas raíces. Si coinciden, devuelve `False` sin tocar contador ni tamaños. Si difieren, enlaza una raíz debajo de la otra, actualiza el tamaño de la nueva raíz, disminuye componentes y devuelve `True`.

Nunca se enlaza directamente `a` debajo de `b`: esos nodos podrían no ser raíces y crear una representación inconsistente. `union(a,a)` es siempre redundante.

```text
raíces distintas → fusionar → componentes − 1 → True
misma raíz       → conservar estado             → False
```

### Comprueba tu comprensión

**Pregunta:** ¿Qué error aparece si union enlaza nodos arbitrarios sin encontrar primero sus raíces?

**Responde esta pregunta en notebook.md.**


## 12. Invariantes

Invariantes esenciales:

1. cada padre es un índice válido;
2. toda cadena termina en una raíz;
3. una raíz es su propio padre;
4. misma raíz equivale a misma componente;
5. el tamaño se consulta en la raíz;
6. componentes coincide con el número de raíces;
7. unión efectiva resta uno; redundante no cambia nada;
8. no existen ciclos salvo el lazo raíz→raíz.

Diagnóstico: `padre=[1,0,2]` es inválido porque 0 y 1 forman un ciclo y `find` nunca alcanza una raíz. Una prueba útil no solo comprueba el retorno de `union`; también revisa contador, conectividad y tamaño.

### Comprueba tu comprensión

**Pregunta:** ¿Qué invariantes viola padre = [1, 0, 2]?

**Responde esta pregunta en notebook.md.**


## 13. Compresión de caminos

Después de encontrar la raíz podemos conectar directamente con ella todos los nodos visitados.

```python
if padre[x] != x:
    padre[x] = find(padre[x])
return padre[x]
```

Antes: `4→3→1→0`. Después de `find(4)`, los nodos del camino apuntan a 0. Cambia la representación interna y disminuye el trabajo futuro; no cambian las componentes, tamaños lógicos ni contador.

<div style="border-left:5px solid #7b4bb7;color:#111111;background:#f4effa;padding:12px 16px;margin:14px 0"><strong style="color:#111111;">IMPORTANTE — Mismo significado, forma más plana</strong><br>La compresión es una optimización de representación, no una operación de unión.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué cambia y qué permanece igual durante la compresión de caminos?

**Responde esta pregunta en notebook.md.**


## 14. Unión por tamaño

Para evitar árboles profundos colocamos la raíz del árbol pequeño debajo de la raíz del grande. Si los tamaños son 4 y 2, la raíz de tamaño 2 pasa a apuntar a la de tamaño 4.

```python
if tamano[raiz_a] < tamano[raiz_b]:
    raiz_a, raiz_b = raiz_b, raiz_a
padre[raiz_b] = raiz_a
tamano[raiz_a] += tamano[raiz_b]
```

Solo el tamaño de la nueva raíz es vigente. Esta clase usa tamaño y no rango para mantener una única heurística. Combinada con compresión, cada operación tiene costo amortizado O(α(n)), casi constante en la práctica.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué colocar el árbol pequeño debajo del grande limita el crecimiento de altura?

**Responde esta pregunta en notebook.md.**


## 15. Ejecución manual de Union-Find

Traza `union(0,1)`, `union(2,3)`, `union(1,3)`, `union(0,3)` y `union(4,5)` sobre seis elementos.

| Operación | raíces | efectiva | padres | tamaños en raíces | componentes |
| --- | --- | --- | --- | --- | ---: |
| inicio | — | — | `[0,1,2,3,4,5]` | seis unos | 6 |
| union(0,1) | 0 / 1 | sí | `[0,0,2,3,4,5]` | raíz 0:2 | 5 |
| union(2,3) | 2 / 3 | sí | `[0,0,2,2,4,5]` | raíz 2:2 | 4 |
| union(1,3) | ? | ? | ? | ? | ? |
| union(0,3) | ? | ? | ? | ? | ? |

Completa la tabla y ejecuta `find(3)`, `tamano_componente(1)` y `conectados(0,4)`. Luego compara con las cinco secuencias visuales.

### Comprueba tu comprensión

**Pregunta:** ¿Qué devuelve union(0, 3) después de unir las componentes {0,1} y {2,3}?

**Responde esta pregunta en notebook.md.**


In [ ]:
from pathlib import Path
import sys
candidatas=[Path.cwd(),Path.cwd().parent,Path.cwd()/'clase_18',Path.cwd().parent/'clase_18']
RAIZ_CLASE=next((r for r in candidatas if (r/'src'/'visualizador_union_find.py').exists()),None)
if RAIZ_CLASE is None:
    raise FileNotFoundError('Abre desde curso-alumnos o clase_18/notebooks')
sys.path.insert(0,str(RAIZ_CLASE))
from src.visualizador_union_find import diagnosticar_widgets, mostrar_union_find
print(diagnosticar_widgets())
mostrar_union_find()


## 16. Algoritmo de Kruskal

Ahora la estructura encuentra su aplicación. Kruskal copia y ordena las aristas; crea Union-Find; recorre de menor a mayor; acepta solo cuando `union(u,v)` devuelve `True`; suma el peso; y termina al elegir V−1 aristas.

```text
ordenadas = copia de aristas ordenada por peso
para cada (u,v,peso):
    si union(u,v):
        aceptar y sumar
    si seleccionadas == V−1: terminar
si faltan aristas: desconectado
```

Ordenar una copia protege la entrada. Llamar `conectados` antes de `union` duplicaría búsquedas de raíces. Las aristas negativas aparecen primero y son válidas.

### Comprueba tu comprensión

**Pregunta:** ¿Cómo usa Kruskal el booleano devuelto por union?

**Responde esta pregunta en notebook.md.**


## 17. Ejecución manual de Kruskal

En el grafo conductor, el orden es C–D:1, A–C:2, D–E:2, B–D:3, A–B:4, A–D:5, C–E:6.

| Paso | Arista | raíces | decisión | elegidas | costo |
| ---: | --- | --- | --- | --- | ---: |
| 1 | C–D:1 | C / D | aceptar | C–D | 1 |
| 2 | A–C:2 | A / C | aceptar | + A–C | 3 |
| 3 | D–E:2 | ? | ? | ? | ? |
| 4 | B–D:3 | ? | ? | ? | ? |

Al llegar a cuatro aristas aceptadas tenemos V−1 y podemos detenernos. La visualización adicional del triángulo sí muestra una arista explícitamente rechazada por ciclo.

### Comprueba tu comprensión

**Pregunta:** ¿Cuál es el costo final del ejemplo conductor y por qué se detiene después de cuatro aristas?

**Responde esta pregunta en notebook.md.**


In [ ]:
from src.visualizador_kruskal import mostrar_kruskal
mostrar_kruskal()


## 18. Grafo desconectado

Si solo existen 0–1 y 2–3, Kruskal termina con dos árboles. Las aristas disponibles se agotaron y nunca se alcanzaron V−1 selecciones. No existe árbol de expansión que incluya los cuatro vértices; el resultado evaluado es `None`.

Lo elegido sí forma un **bosque de expansión mínima**, pero la función no necesita devolverlo. En Road Reparation, `None` se traduce a `IMPOSSIBLE`.

<div style="border-left:5px solid #b42318;color:#111111;background:#fff0ef;padding:12px 16px;margin:14px 0"><strong style="color:#111111;">PRECAUCIÓN — Finalizar no significa éxito</strong><br>Después del bucle siempre comprueba si fueron elegidas exactamente V−1 aristas.</div>

### Comprueba tu comprensión

**Pregunta:** ¿Qué condición permite distinguir un MST completo de un bosque desconectado?

**Responde esta pregunta en notebook.md.**


## 19. Pesos negativos, empates y casos especiales

Kruskal acepta pesos negativos: ordenar sigue siendo válido y una conexión negativa es especialmente conveniente. Dijkstra no los admite porque una distancia considerada puede mejorar mediante un camino posterior.

Aristas paralelas se consideran por costo; una posterior suele rechazarse. Una autoarista produce `union(u,u) == False`. Con pesos empatados puede haber varios MST válidos: las pruebas deben verificar costo, V−1 aristas, conectividad, ausencia de ciclos y pertenencia al grafo, no una lista exacta.

Cero o un vértice requieren cero aristas y devuelven `(0.0, [])`.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué un test con pesos empatados no debe exigir siempre una lista exacta de aristas?

**Responde esta pregunta en notebook.md.**


## 20. Complejidad

Ordenar E aristas cuesta O(E log E). Cada `find` o `union`, con las dos heurísticas, cuesta O(α(V)) amortizado. No necesitamos desarrollar la función inversa de Ackermann: para tamaños prácticos se comporta casi como una constante.

```text
copiar/validar     O(E)
ordenar            O(E log E)   ← domina
uniones            O(E α(V))
total              O(E log E)
```

También se escribe O(E log V) en este contexto, pero usaremos O(E log E) como fórmula principal. La memoria adicional es O(V+E) por la copia ordenada, arreglos y salida.

### Comprueba tu comprensión

**Pregunta:** ¿Qué parte domina la complejidad total de Kruskal?

**Responde esta pregunta en notebook.md.**


## 21. CSES Road Reparation

[CSES Road Reparation](https://cses.fi/problemset/task/1675/) pide reparar carreteras bidireccionales para conectar todas las ciudades con costo mínimo. No hay origen: es un problema de MST.

CSES numera ciudades 1…n; nuestra función reutilizable usa 0…n−1, por lo que al leer convertimos `a-1` y `b-1`. Si `costo_reparacion` devuelve `None`, imprimimos `IMPOSSIBLE`; en otro caso imprimimos el entero. El ejemplo oficial tiene cinco ciudades, seis carreteras y costo mínimo 14.

Actividad guiada: modela lista de aristas, explica la conversión, predice qué carretera cierra un ciclo y diseña una prueba desconectada.

### Comprueba tu comprensión

**Pregunta:** ¿Qué dos adaptaciones separan el formato de CSES de nuestra función reutilizable?

**Responde esta pregunta en notebook.md.**


## 22. LeetCode Redundant Connection

[LeetCode 684 — Redundant Connection](https://leetcode.com/problems/redundant-connection/) reutiliza Union-Find sin ordenar aristas. Se procesan en el orden dado; la primera cuyo `union(u,v)` devuelve `False` es redundante porque cierra un ciclo.

Este problema es opcional y no forma parte de la evaluación obligatoria. Sirve para observar que Union-Find es una estructura reutilizable: Kruskal es una aplicación, no su definición completa.

Otras extensiones opcionales: comparar con DFS, investigar Prim, dibujar una compresión o construir dos MST distintos con el mismo costo.

### Comprueba tu comprensión

**Pregunta:** ¿Qué cambia entre Redundant Connection y Kruskal aunque ambos usen Union-Find?

**Responde esta pregunta en notebook.md.**


## 23. Implementación

Copia `src/union_find_kruskal_base.py` como `implementacion.py` en tu entrega y conserva firmas. Orden recomendado:

1. validar cantidad e índices;
2. inicializar padres, tamaños y contador;
3. `find` sin y luego con compresión;
4. `union` por tamaño y consultas;
5. normalizar una copia de aristas;
6. Kruskal y detección de desconexión;
7. `costo_reparacion`.

Los pesos de Kruskal aceptan `int|float` finitos, no `bool`, y se normalizan a `float`. Los extremos son enteros dentro de rango. La entrada nunca se ordena in-place.

### Comprueba tu comprensión

**Pregunta:** ¿Qué responsabilidades deben estar probadas antes de integrar Union-Find dentro de Kruskal?

**Responde esta pregunta en notebook.md.**


## 24. Pruebas

Debes escribir al menos seis pruebas propias explicadas: unión efectiva, repetida, transitividad, tamaño, arista rechazada por ciclo y grafo desconectado. Cada una documenta regla, entrada, resultado y razón.

Complementa con índice negativo, peso negativo, paralelas, autoarista, empate, un vértice, no mutación y compresión. Con empates valida propiedades del resultado.

```python
def test_union_repetida_no_reduce_componentes():
    uf = UnionFind(2)
    assert uf.union(0, 1)
    antes = uf.numero_componentes()
    assert not uf.union(1, 0)
    assert uf.numero_componentes() == antes
```

Ejecuta públicas y propias con `evaluar.py` y conserva la salida completa en el reporte.

### Comprueba tu comprensión

**Pregunta:** ¿Qué invariante protege una prueba de unión repetida?

**Responde esta pregunta en notebook.md.**


## 25. Cierre hacia ordenamiento topológico

La progresión queda así:

| Necesidad repetida | Estructura |
| --- | --- |
| orden de descubrimiento | cola |
| costo 0 antes que costo 1 | deque |
| menor distancia tentativa | heap |
| pertenencia a la misma componente | Union-Find |

Union-Find mantiene componentes y detecta ciclos en grafos no dirigidos. En la siguiente clase cambiará el contrato: aristas dirigidas representarán dependencias y necesitaremos construir un orden válido.

```text
grafo dirigido → dependencias → grados de entrada → cola → orden topológico
```

No implementaremos todavía el algoritmo de Kahn.

### Comprueba tu comprensión

**Pregunta:** ¿Qué estructura se necesitaría para procesar tareas cuando unas dependen de otras?

**Responde esta pregunta en notebook.md.**
